<a href="https://colab.research.google.com/github/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is **Ranking Improvement**.

I am framing this as a **ranking/scoring problem**. The goal is to estimate a performance-related score for each content item so that the score can support better ranking decisions.

The initial idea is to predict a content item's observed **CTR (`ctr`)** using available search, competition, content, and engagement-related signals.

This is not simply a classification problem because the goal is not only to predict whether an item is good or bad. The goal is to estimate a continuous performance signal that can be used to prioritize or compare content items.

The final task framing may change after exploratory analysis and data-quality checks.

In [ ]:
import pandas as pd

import pandas as pd

url = "https://raw.githubusercontent.com/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())


Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

My initial target/proxy is **CTR (`ctr`)**.

CTR is an observed user-interaction signal in the starter dataset. I will investigate whether available content and search-related features can help estimate CTR.

The target is therefore:

**features about a content item → predicted CTR**

CTR is only a proxy for search performance. A click does not necessarily mean that the user found the content useful or relevant.

I will also investigate `avg_position` carefully. Because it represents existing ranking position, using it directly as a feature could cause the model to reproduce existing ranking behavior rather than identify independent signals for ranking improvement.

In [ ]:
print("Target column: ctr")
print("Data type:", df["ctr"].dtype)
print("Missing values:", df["ctr"].isna().sum())

print("Mean CTR:", df["ctr"].mean())
print("Median CTR:", df["ctr"].median())

display(df[["content_id", "ctr"]].head(10))

Target column: ctr
Data type: float64
Missing values: 0
Mean CTR: 0.5107333333333334
Median CTR: 0.07


,content_id,ctr
0,content_304f48230142,0.76
1,content_a1fb4e703a9e,0.05
2,content_9aa793d4d895,0.09
3,content_331d6c4de07b,0.49
4,content_d99b7a2d90ca,0.13
5,content_d4084a4bc775,0.03
6,content_9a34b442b552,0.00
7,content_a63219c6e95a,0.06
8,content_5e6c160719bc,0.09
9,content_c27558df2b0c,0.16


## 3. Success metric

My initial prediction metric will be **Mean Absolute Error (MAE)**.

MAE measures the average absolute difference between the predicted CTR and the observed CTR. A lower MAE means that the predictions are closer to the observed values.

Because my final use case is ranking improvement, prediction error alone is not enough. I will also investigate a ranking-oriented metric such as **NDCG@K** after establishing a suitable baseline.

I will compare the ML approach with a simple baseline before claiming that ML provides improvement.

I will not assume that an accuracy threshold such as 70% is meaningful because this is not currently framed as a binary classification problem.

In [ ]:
print("CTR summary:")
print(df["ctr"].describe())

CTR summary:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content item observation**.

Each row represents a content item associated with a client and contains information about search demand, competition, content characteristics, ranking, and observed performance.

Important fields include:

- `content_id`
- `client_id`
- `search_volume`
- `competition`
- `content_type`
- `main_intent`
- `word_count`
- `ctr`
- `avg_position`
- `engagement_rate`

The dataframe below shows actual observations from the starter dataset rather than manually created example data.

In [ ]:
unit_columns = [
    "content_id",
    "search_volume",
    "competition",
    "content_type",
    "main_intent",
    "word_count",
    "ctr",
    "avg_position",
    "engagement_rate"
]

unit_df = df[unit_columns].head(10)

display(unit_df)

,content_id,search_volume,competition,content_type,main_intent,word_count,ctr,avg_position,engagement_rate
0,content_304f48230142,10.0,0.67,keyword article,transactional,3221.0,0.76,10.6,5.88
1,content_a1fb4e703a9e,90.0,0.01,keyword article,informational,2481.0,0.05,20.3,0.00
2,content_9aa793d4d895,0.0,0.00,keyword article,informational,3515.0,0.09,36.5,0.00
3,content_331d6c4de07b,10.0,0.00,keyword article,commercial,NaN,0.49,6.2,1.28
4,content_d99b7a2d90ca,0.0,0.00,keyword article,informational,2803.0,0.13,44.0,0.00
5,content_d4084a4bc775,720.0,1.00,keyword article,transactional,3080.0,0.03,8.5,0.00
6,content_9a34b442b552,0.0,0.00,keyword article,informational,3059.0,0.00,7.0,0.00
7,content_a63219c6e95a,590.0,0.44,keyword article,commercial,NaN,0.06,21.2,3.57
8,content_5e6c160719bc,0.0,0.00,keyword article,informational,3807.0,0.09,46.0,5.88
9,content_c27558df2b0c,0.0,0.00,keyword article,informational,NaN,0.16,4.9,0.00


In [ ]:
target_view = df[[
    "content_id",
    "ctr"
]].head(10)

display(target_view)

,content_id,ctr
0,content_304f48230142,0.76
1,content_a1fb4e703a9e,0.05
2,content_9aa793d4d895,0.09
3,content_331d6c4de07b,0.49
4,content_d99b7a2d90ca,0.13
5,content_d4084a4bc775,0.03
6,content_9a34b442b552,0.00
7,content_a63219c6e95a,0.06
8,content_5e6c160719bc,0.09
9,content_c27558df2b0c,0.16


## 5. Why ML beats a fixed rule here

A fixed rule might rank content only by one signal, such as search volume or CTR.

However, content performance may depend on multiple factors including search demand, competition, content type, search intent, content length, existing position, and engagement.

A fixed rule may be useful as a simple baseline, but it cannot easily combine these signals or learn patterns from historical observations.

ML may be useful because it can combine multiple features and learn predictive relationships from the available data.

However, I will not assume that ML is automatically better. I will compare an ML approach against simple baseline rules. If a simple rule performs similarly or better, the simpler approach may be preferable.

The purpose of ML is therefore **decision support for ranking improvement**, not simply training a model because ML is available.

In [ ]:
# Simple fixed-rule example:
# Rank content only by search volume.

rule_columns = [
    "content_id",
    "search_volume",
    "ctr"
]

rule_df = df[rule_columns].dropna().sort_values(
    "search_volume",
    ascending=False
)

print("Top 10 content items by search volume:")
display(rule_df.head(10))

Top 10 content items by search volume:


,content_id,search_volume,ctr
12140,content_ef99c4abd9ab,74000.0,0.03
6972,content_bf67a444faef,60500.0,0.00
28282,content_454cc6654c6e,60500.0,0.00
17907,content_5ec29ae79c60,60500.0,0.00
18701,content_deb54e9e19cd,60500.0,0.00
22788,content_ee4630879d03,49500.0,0.00
8055,content_cd6760921db8,49500.0,0.00
16005,content_83e3da1394ac,49500.0,0.00
13502,content_f76ccf7a7834,49500.0,0.15
2815,content_7868341d97dd,40500.0,0.08


In [ ]:
# Compare CTR for higher-search-volume and lower-search-volume groups.
# This is descriptive only and does not prove causation.

median_search_volume = df["search_volume"].median()

high_volume_ctr = df.loc[
    df["search_volume"] >= median_search_volume,
    "ctr"
].mean()

low_volume_ctr = df.loc[
    df["search_volume"] < median_search_volume,
    "ctr"
].mean()

print("Median search volume:", median_search_volume)
print("Mean CTR for higher-volume content:", high_volume_ctr)
print("Mean CTR for lower-volume content:", low_volume_ctr)

## Self-check

- **Task type identified:** Yes. I have framed the problem as a ranking/scoring task.
- **Target/proxy identified:** Yes. My initial target is `ctr`.
- **Success metric identified:** Yes. I will initially use MAE and investigate a ranking-oriented metric such as NDCG@K.
- **Unit of analysis identified:** Yes. One row represents one content item observation.
- **Real dataframe shown:** Yes. The dataframe comes directly from `content_refresh_anonymized.csv`.
- **Target shown:** Yes. The notebook displays actual `content_id` and `ctr` values.
- **Action identified:** The output can support prioritization and ranking analysis or controlled experiments.
- **Why ML:** ML may combine multiple signals better than a single fixed rule, but it must be compared against baselines.
- **Careful claims:** I am not claiming causation or that ML will definitely improve ranking.
- **Data privacy:** No client names, private queries, or sensitive information are intentionally included.
- **Execution:** The notebook should run from top to bottom without errors.
- **Repository:** The executed notebook will be committed under `work/notebooks/w02_ml_task_framing.ipynb`.